# GENE linear scan

Point `SCAN_DIR` at a `scanfiles*` folder and run. `load_scan` reads `scan.log`
and every run's `nrg`, and hands back one table: a column per scanned parameter,
plus `gamma`, `omega`, `Qes_e/Qes_i` and `Qem/Qes_e`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import matplotlib.pyplot as plt

for _p in (Path.cwd(), *Path.cwd().parents[:2]):
    if (_p / "scripts" / "scan_summary.py").is_file():
        sys.path.insert(0, str(_p / "scripts"))
        break
from scan_summary import load_scan

SCAN_DIR = "/path/to/scanfiles0001"   # <-- edit me

df = load_scan(SCAN_DIR)
ky, groups = df.attrs["ky"], df.attrs["group_by"]
print(f"{len(df)} runs — ky column {ky!r}, grouped by {groups or 'nothing'}")
df

## Plot

One curve per parameter combination. Change the column name to plot anything
else in the table.

In [ ]:
def plot(column, ylabel=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    for key, sub in (df.groupby(groups) if groups else [(None, df)]):
        label = ", ".join(f"{p}={v:g}" for p, v in
                          zip(groups, key if isinstance(key, tuple) else (key,))) \
                if groups else None
        ax.plot(sub[ky], sub[column], "o-", ms=3, label=label)
    ax.set_xlabel(r"$k_y \rho_s$")
    ax.set_ylabel(ylabel or column)
    ax.grid(True, alpha=0.3)
    if groups:
        ax.legend(fontsize=8)
    return ax

plot("gamma", r"$\gamma$")
plot("omega", r"$\omega$")
plot("Qes_e/Qes_i")
plot("Qem/Qes_e")
plt.show()

## Anything else

`df` is a plain DataFrame — filter, pivot or export it however you like.

In [ ]:
df.to_csv("scan_summary.csv", index=False)